[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/03_install_access/03_setup_check.ipynb)


# 03 — 환경 설치·검증

> 본문: [`03_install_access.md`](03_install_access.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 `data/` 의 **실제 BoltzGen 실행 결과**에서 계산합니다(임의값 아님).
> **분석 셀 실행 19초.**

> 라이브 `boltzgen run` 셀은 NVIDIA GPU 에서 동작합니다(CPU 폴백 없음) — Colab 이면 **런타임 → T4 GPU**.
> 그 셀을 건너뛰어도 나머지는 그대로 진행됩니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 노트북은 **Colab과 로컬 모두**에서 동작합니다.

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`03_install_access`) 폴더로 자동 이동한 뒤, `data/` 의 실제 결과로 실습합니다.
- **로컬**: 이 노트북을 `03_install_access/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

> 런타임은 **기본값 그대로** 두면 됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "03_install_access"
PIP_PKGS = ""   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# 필요한 라이브러리 확보: import 안 되는 것만 설치(Colab 새 런타임/로컬 모두 자동 대응)
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# ── 내가 만든 결과 우선, 없으면 레퍼런스 ──────────────────────────────────────
#   MY  : 이 노트북에서 내가 직접 돌려 만든 산출물
#   REF : 저장소에 커밋된 레퍼런스 결과(대조군 · 실행을 건너뛰어도 실습이 이어지도록)
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """산출물 하나를 찾는다 — 내가 만든 my_run/ 트리를 먼저 뒤지고, 없으면 레퍼런스 폴더에서.

    파일명 규칙이 달라도(내 실행 rank1_*.cif / 레퍼런스 rank001_*.cif,
    final_<budget>_designs / final_designs) 같은 글롭으로 잡히도록 설계했습니다.
    """
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 이미 설계를 돌렸다면 그 결과를 이어받는다(이 챕터에서 다시 안 돌려도 됨).

    내 my_run/ 에 결과가 있으면 그대로 쓰고, 없으면 인자로 준 순서대로 앞 챕터를 찾는다.
    아무 데도 없으면 MY 를 그대로 둬서 find_one() 이 레퍼런스로 폴백하게 한다.
    """
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 내가 만든 그림은 my_ 접두어로 저장 — 저장소에 커밋된 레퍼런스 그림을 덮어쓰지 않도록.
def my_fig(name):
    return f"my_{name}"

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd())

## 1) BoltzGen 설치 (Colab) — 본문 3.2
Colab 무료 GPU(T4 등)에서도 설치됩니다. 가중치(~6GB)는 첫 `run` 때 자동 다운로드됩니다.

In [ ]:
import sys
if IN_COLAB:
    _run(f'"{sys.executable}" -m pip -q install boltzgen')
    # cuequivariance 커널은 cuBLAS >= 12.5 를 요구(본문 3.3). Colab 이미지에 따라 정합 보강:
    _run(f'"{sys.executable}" -m pip -q install "nvidia-cublas-cu12>=12.5" || true')
else:
    print("로컬: 이미 설치된 boltzgen_env 등을 사용하세요 (본문 3.2).")

## 2) 드라이버 CUDA 상한 — `nvidia-smi` (본문 3.3)
우상단 `CUDA Version: 12.x` 가 **드라이버가 지원하는 상한**입니다.

In [ ]:
!nvidia-smi | head -n 12

## 3) PyTorch CUDA / GPU 인식
`torch.version.cuda` 가 드라이버 상한 이하의 12.x 이고 `cuda available: True` 면 정상.

In [ ]:
try:
    import torch
    print("torch:", torch.__version__, "| built cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        a = torch.randn(256, 256, device="cuda"); b = torch.randn(256, 256, device="cuda")
        print("device:", torch.cuda.get_device_name(0), "| matmul ok:", bool(torch.isfinite((a @ b).sum())))
except Exception as e:
    print("torch import/실행 실패:", repr(e)[:160])
    print("→ Colab이면 [런타임 재시작] 후 재실행, 로컬이면 본문 3.3 CUDA 정합(드라이버↔torch↔cuBLAS) 확인")

## 3.5) 실행 로그 미리 읽기 — 커널·배치 (본문 3.1.5)
BoltzGen은 **compute capability 8 이상**에서만 가속 커널을 켭니다(그 미만이면 자동으로 끈 채 정상 실행).
아래 셀이 내 GPU 에서 커널이 켜지는지, 어떤 diffusion 배치가 쓰일지 미리 알려줍니다.

In [ ]:
try:
    import torch
    if not torch.cuda.is_available():
        print("GPU 없음 → 라이브 설계(boltzgen run)는 불가합니다 (CPU 폴백 없음).")
        print("   단, 분석·해석 셀은 그대로 실습할 수 있어요(레퍼런스 결과로 이어집니다).")
    else:
        cap = torch.cuda.get_device_capability()
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU: {torch.cuda.get_device_name(0)} | capability {cap} | VRAM {vram:.1f} GB")
        print("가속 커널:", "ON" if cap[0] >= 8 else "OFF (자동 비활성 — 정상 동작, 조금 느림)")
        print("\n[이 GPU에서의 실행 팁]")
        print("  · --num_designs 4~30 이면 diffusion 배치가 자동으로 1 — 실습 규모엔 옵션이 필요 없어요.")
        if vram < 14:
            print("  · 100개 이상 뽑을 땐 --diffusion_batch_size 1 을 명시하세요(기본값이 10이라 메모리가 큽니다).")
        if cap[0] < 8:
            print("  · bf16 미지원 카드(T4/V100)에서 정밀도 오류가 나면: --config folding trainer.precision=32")
except Exception as e:
    print("진단 실패:", repr(e)[:160])

## 4) cuequivariance 가속 커널
실패 시 `undefined symbol: cublasGemmGroupedBatchedEx` → cuBLAS 12.5+ 로 보강(본문 3.3 케이스 ②).

> capability 8 미만 GPU(T4 등)는 이 커널을 **애초에 쓰지 않으므로** 이 오류가 나지 않아요. 실패해도 설계는 돌아갑니다.

In [ ]:
try:
    from cuequivariance_ops_torch import triangle_multiplicative_update
    print("cuequivariance kernel: OK")
except Exception as e:
    print("FAILED:", repr(e)[:160])
    print("→ pip install 'nvidia-cublas-cu12>=12.5' 후 런타임 재시작")

## 5) BoltzGen CLI 확인

In [ ]:
import shutil
if shutil.which("boltzgen"):
    !boltzgen --version
    !boltzgen --help 2>&1 | sed -n '1,14p'
else:
    print("boltzgen 미설치 — 1) 셀을 먼저 실행하세요.")

## 6) 설계 명세 검증 + (선택) 스모크 테스트 — 본문 3.6
예제는 boltzgen 레포에 있으므로, 라이브 검증을 원하면 레포를 클론합니다.

In [ ]:
import shutil, pathlib
if IN_COLAB and shutil.which("boltzgen") and not pathlib.Path("boltzgen-src").exists():
    _run("git clone --depth 1 https://github.com/HannesStark/boltzgen.git boltzgen-src")
spec = "boltzgen-src/example/vanilla_protein/1g13prot.yaml"
if shutil.which("boltzgen") and pathlib.Path(spec).exists():
    !boltzgen check {spec}
    print("\n→ 6스텝을 실제로 돌려보는 건 Ch.04 노트북에서 합니다(num_designs 4, 스모크 규모).")
else:
    print("건너뜀: boltzgen/예제 미준비. 위 1) 설치 후 다시 실행하세요.")